# Multi-Source Emotion Dataset Cleaning & Split Pipeline

## Sources

1. **GoEmotions**  
   Path:  
   `/kaggle/input/datasets/debarshichanda/goemotions`

2. **ISEAR**  
   Source: HuggingFace Dataset  
   `gsri-18/ISEAR-dataset-complete`  
   *(auto-downloaded via `datasets` library)*

3. **EmpatheticDialogues**  
   Path:  
   `/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv`

---

## Emotion Mapping → 5 Project Classes

### 1. Sad

- **GoEmotions:** sadness, grief, remorse, disappointment, caring  
- **ISEAR:** sadness, shame, guilt  
- **EmpatheticDialogues:** sad, devastated, lonely, disappointed

### 2. Angry

- **GoEmotions:** anger, annoyance, disgust  
- **ISEAR:** anger, disgust  
- **EmpatheticDialogues:** angry, furious, disgusted, annoyed

### 3. Joyful

- **GoEmotions:** joy, excitement, pride, gratitude, optimism, amusement, admiration  
- **ISEAR:** joy  
- **EmpatheticDialogues:** joyful, excited, proud, grateful, hopeful, impressed

### 4. Anxious

- **GoEmotions:** fear, nervousness  
- **ISEAR:** fear  
- **EmpatheticDialogues:** anxious, afraid, terrified, apprehensive

### 5. Content

- **GoEmotions:** love, relief, approval, desire, curiosity  
- **ISEAR:** *(none — no direct content class)*  
- **EmpatheticDialogues:** content, confident, prepared, sentimental, caring, trusting

---

## Outputs

All files are saved to:

`/kaggle/working/`

| File Name | Description |
|----------|-------------|
| `cleaned_goemotions.csv` | GoEmotions only, cleaned & balanced |
| `cleaned_isear.csv` | ISEAR only, cleaned & balanced |
| `cleaned_empdial.csv` | EmpatheticDialogues only, cleaned & balanced |
| `combined_all.csv` | All three datasets combined & balanced (~15k rows) |
| `train.csv` | 80% of combined dataset |
| `val.csv` | 10% of combined dataset |
| `test.csv` | 10% of combined dataset |

---

## Pipeline Summary

1. Load datasets from 3 different sources  
2. Map original emotions into 5 unified classes  
3. Clean text data  
4. Balance class distribution  
5. Merge all datasets  
6. Split into Train / Validation / Test sets  
7. Export CSV files for training

In [1]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))
            
#  Download NLTK resources 
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# CONFIG 
GOEMOTIONS_PATH = "/kaggle/input/datasets/debarshichanda/goemotions/data/full_dataset/"
EMPDIAL_PATH    = "/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv"
OUTPUT_DIR      = "/kaggle/working/"
 
MIN_TEXT_LEN = 10    # characters
MAX_TEXT_LEN = 300   # characters
 
# Samples per class per source — keeps sources balanced before combining
# GoEmotions: large source  → 2000/class
# ISEAR:      small source  → take all available (≈600–900/class after filter)
# EmpDial:    medium source → 1500/class
GOEMOTIONS_PER_CLASS = 2000
EMPDIAL_PER_CLASS    = 1500
# ISEAR uses all available rows per class (no cap) since it's small
 
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
 
RANDOM_SEED = 42

/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv
/kaggle/input/datasets/debarshichanda/goemotions/tables/emotion_words.csv
/kaggle/input/datasets/debarshichanda/goemotions/data/ekman_labels.csv
/kaggle/input/datasets/debarshichanda/goemotions/data/full_dataset/goemotions_2.csv
/kaggle/input/datasets/debarshichanda/goemotions/data/full_dataset/goemotions_1.csv
/kaggle/input/datasets/debarshichanda/goemotions/data/full_dataset/goemotions_3.csv


In [2]:
# EMOTION MAPPING
GOEMOTIONS_MAP = {
    "sadness":        "sad",
    "grief":          "sad",
    "remorse":        "sad",
    "disappointment": "sad",
    "caring":         "sad",
    "anger":          "angry",
    "annoyance":      "angry",
    "disgust":        "angry",
    "joy":            "joyful",
    "excitement":     "joyful",
    "pride":          "joyful",
    "gratitude":      "joyful",
    "optimism":       "joyful",
    "amusement":      "joyful",
    "admiration":     "joyful",
    "fear":           "anxious",
    "nervousness":    "anxious",
    "love":           "content",
    "relief":         "content",
    "approval":       "content",
    "desire":         "content",
    "curiosity":      "content",
}

ISEAR_MAP = {
    "joy":     "joyful",
    "fear":    "anxious",
    "anger":   "angry",
    "sadness": "sad",
    "disgust": "angry",
    "shame":   "sad",
    "guilt":   "sad",
}

EMPDIAL_MAP = {
    # sad
    "sad":           "sad",
    "devastated":    "sad",
    "lonely":        "sad",
    "disappointed":  "sad",
    "guilty":        "sad",
    "ashamed":       "sad",
    "sentimental":   "sad",     # sentimental leans melancholic
    # angry
    "angry":         "angry",
    "furious":       "angry",
    "disgusted":     "angry",
    "annoyed":       "angry",
    "jealous":       "angry",
    # joyful
    "joyful":        "joyful",
    "excited":       "joyful",
    "proud":         "joyful",
    "grateful":      "joyful",
    "hopeful":       "joyful",
    "impressed":     "joyful",
    "anticipating":  "joyful",
    # anxious
    "anxious":       "anxious",
    "afraid":        "anxious",
    "terrified":     "anxious",
    "apprehensive":  "anxious",
    "surprised":     "anxious",  # surprise often co-occurs with anxiety
    # content
    "content":       "content",
    "confident":     "content",
    "prepared":      "content",
    "caring":        "content",
    "trusting":      "content",
    "faithful":      "content",
    "nostalgic":     "content",  # nostalgic is warm, not sad in EmpDial context
    # Excluded: embarrassed, disillusioned (too ambiguous)
}

In [3]:
# TEXT CLEANING
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
 
# Keep negations and emotion-loaded words even if in stopword list
KEEP_WORDS = {
    "not", "no", "never", "neither", "nor",
    "very", "so", "too", "really", "quite",
    "angry", "sad", "happy", "anxious", "calm",
    "afraid", "worried", "guilty", "lonely"
}
FILTERED_STOP_WORDS = stop_words - KEEP_WORDS
 
 
def clean_text(text: str) -> str:
    """
    1. Lowercase
    2. Remove URLs, @mentions, #hashtags
    3. Remove Reddit artifacts: [removed], [deleted]
    4. Remove special characters (keep letters, digits, spaces, basic punctuation)
    5. Collapse whitespace
    6. Remove stopwords (keeping emotion-relevant ones)
    7. Lemmatize
    Returns empty string if input is invalid.
    """
    if not isinstance(text, str):
        return ""
 
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\[removed\]|\[deleted\]", "", text)
    text = re.sub(r"[^a-z0-9\s'.,!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
 
    tokens = text.split()
    tokens = [t for t in tokens if t not in FILTERED_STOP_WORDS]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
 
    return " ".join(tokens)

In [4]:
#SHARED UTILITIES
def inspect_dataframe(name: str, df: pd.DataFrame):
    print(f"\n── [{name}] columns: {df.columns.tolist()}")
    print(f"   shape: {df.shape}")
    print(df.head(2).to_string())
 
def apply_quality_filters(df: pd.DataFrame, text_col: str, source: str) -> pd.DataFrame:
    """Length bounds + numeric filter + deduplication on raw text."""
    before = len(df)
    df = df[df[text_col].str.len().between(MIN_TEXT_LEN, MAX_TEXT_LEN)]

    def is_mostly_text(text):
        tokens = str(text).split()
        if not tokens:
            return False
        return sum(1 for t in tokens if t.isdigit()) / len(tokens) < 0.5

    df = df[df[text_col].apply(is_mostly_text)]
    df = df.drop_duplicates(subset=[text_col])
    print(f"  [{source}] Quality filter: {before} → {len(df)} rows")
    return df
 
 
def apply_label_mapping(df: pd.DataFrame, label_col: str,
                        mapping: dict, source: str) -> pd.DataFrame:
    """Map raw labels to 5 project classes. Drop unmapped rows."""
    df = df.copy()
    df["project_label"] = df[label_col].str.lower().str.strip().map(mapping)
    before = len(df)
    df = df.dropna(subset=["project_label"])
    print(f"  [{source}] Label mapping: {before} → {len(df)} rows")
    print(f"   Distribution:\n{df['project_label'].value_counts().to_string()}")
    return df
 
 
def apply_cleaning(df: pd.DataFrame, text_col: str, source: str) -> pd.DataFrame:
    df = df.copy()
    df["clean_text"] = df[text_col].apply(clean_text)
    before = len(df)
    df = df[df["clean_text"].str.strip() != ""]
    print(f"  [{source}] After cleaning: {before} → {len(df)} rows")
    return df
 
 
def balance_per_class(df: pd.DataFrame, cap: int, source: str) -> pd.DataFrame:
    """
    Sample up to `cap` rows per class.
    If cap is None, take all available rows (used for small sources like ISEAR).
    """
    parts = []
    for label in df["project_label"].unique():
        subset = df[df["project_label"] == label]
        if cap is not None and len(subset) > cap:
            subset = subset.sample(n=cap, random_state=RANDOM_SEED)
        parts.append(subset)
        print(f"  [{source}] '{label}': {len(subset)} rows")
 
    balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)
    return balanced
 
 
def to_standard(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Reduce to [text, label, source] columns and drop any remaining nulls."""
    out = df[["clean_text", "project_label"]].copy()
    out.columns = ["text", "label"]
    out["source"] = source_name
    out = out.dropna()
    out = out[out["text"].str.strip() != ""]
    return out
 
 
def save_csv(df: pd.DataFrame, filename: str):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"  ✅ Saved: {filename}  ({df.shape[0]} rows)")

In [5]:
# SOURCE LOADERS
def load_goemotions() -> pd.DataFrame:
    """
    GoEmotions ships as train/dev/test CSV files, or a single full_dataset.csv.
    Handles both single-label column and one-hot column formats.
    """
    print("\n" + "=" * 60)
    print("  Loading GoEmotions")
    print("=" * 60)

    dfs = []
    for fname in os.listdir(GOEMOTIONS_PATH):
        if fname.endswith(".csv"):
            fpath = os.path.join(GOEMOTIONS_PATH, fname)
            chunk = pd.read_csv(fpath)
            print(f"  Loaded {fname}: {len(chunk)} rows")
            dfs.append(chunk)

    if not dfs:
        raise FileNotFoundError(f"No CSVs in {GOEMOTIONS_PATH}")

    df = pd.concat(dfs, ignore_index=True)

    # full_dataset has multiple rater rows per text — aggregate by summing votes
    # then each emotion column = number of raters who labeled it
    emotion_cols = [c for c in df.columns if c in GOEMOTIONS_MAP]
    df = df.groupby("text", as_index=False)[emotion_cols].sum()
    print(f"  Total rows after rater aggregation: {len(df)}")

    inspect_dataframe("GoEmotions", df)

    # Detect text column
    text_col = None
    for c in ["text", "comment_text", "sentence", "utterance"]:
        if c in df.columns:
            text_col = c
            break
    if not text_col:
        text_col = [c for c in df.columns if df[c].dtype == object][0]
    print(f"  Text column: '{text_col}'")

    # Detect label format: single column vs one-hot
    label_col = None
    for c in ["emotion", "label", "primary_emotion", "emotion_label"]:
        if c in df.columns:
            label_col = c
            break

    if label_col:
        df = apply_label_mapping(df, label_col, GOEMOTIONS_MAP, "GoEmotions")
    else:
        # One-hot format: pick highest-scoring mapped emotion
        present = [e for e in GOEMOTIONS_MAP if e in df.columns]
        if not present:
            raise ValueError("GoEmotions: cannot detect label format.")

        def dominant(row):
            scores = {e: row[e] for e in present}
            top = max(scores, key=scores.get)
            return GOEMOTIONS_MAP.get(top)

        df = df.copy()
        df["project_label"] = df.apply(dominant, axis=1)
        before = len(df)
        df = df.dropna(subset=["project_label"])
        print(f"  [GoEmotions] One-hot label mapping: {before} → {len(df)} rows")

    df = apply_quality_filters(df, text_col, "GoEmotions")
    df = apply_cleaning(df, text_col, "GoEmotions")
    df = balance_per_class(df, cap=GOEMOTIONS_PER_CLASS, source="GoEmotions")
    return to_standard(df, "goemotions")
 
 
def hf_login():
    """
    Authenticate with HuggingFace Hub using the HF_TOKEN stored in Kaggle secrets.
    This speeds up dataset downloads and avoids anonymous rate limits.
 
    To set this up in Kaggle:
        Notebook → Add-ons → Secrets → Add secret
        Name: HF_TOKEN  |  Value: hf_xxxxxxxxxxxxxxxxx
    """
    try:
        from huggingface_hub import login
        from kaggle_secrets import UserSecretsClient
 
        token = UserSecretsClient().get_secret("HF_TOKEN")
        login(token=token, add_to_git_credential=False)
        print("  ✅ HuggingFace login successful via Kaggle secret HF_TOKEN.")
    except Exception as e:
        print(f"  ⚠️  HuggingFace login skipped: {e}")
        print("      Proceeding anonymously — may be slower or rate-limited.")
 
 
def load_isear() -> pd.DataFrame:
    """
    Load ISEAR from HuggingFace Hub using the datasets library.
    Authenticates via Kaggle secret HF_TOKEN before downloading.
    Expected columns: 'text' (or 'sentence'), 'label' (or 'emotion')
    """
    print("\n" + "=" * 60)
    print("  Loading ISEAR (HuggingFace Hub)")
    print("=" * 60)
 
    hf_login()
 
    try:
        from datasets import load_dataset
        raw = load_dataset("gsri-18/ISEAR-dataset-complete")
        # Merge all splits
        splits = list(raw.keys())
        print(f"  Available splits: {splits}")
        dfs = [raw[s].to_pandas() for s in splits]
        df = pd.concat(dfs, ignore_index=True)
        print(f"  Total rows: {len(df)}")
        inspect_dataframe("ISEAR", df)
    except Exception as e:
        raise RuntimeError(
            f"Could not load ISEAR from HuggingFace Hub: {e}\n"
            "Make sure 'datasets' is installed: !pip install datasets -q\n"
            "Or check your internet connection in Kaggle settings."
        )
 
    # Detect text and label columns
    text_col = None
    for c in ["content","text", "sentence", "TEXT", "Sentence"]:
        if c in df.columns:
            text_col = c
            break
    if not text_col:
        text_col = [c for c in df.columns if df[c].dtype == object][0]
 
    label_col = None
    for c in ["label", "emotion", "Label", "Emotion", "SIT_CAT"]:
        if c in df.columns:
            label_col = c
            break
    if not label_col:
        raise ValueError(f"ISEAR: cannot find label column. Columns: {df.columns.tolist()}")
 
    print(f"  Text column: '{text_col}', Label column: '{label_col}'")
    print(f"  Unique labels: {df[label_col].unique().tolist()}")
 
    df = apply_label_mapping(df, label_col, ISEAR_MAP, "ISEAR")
    df = apply_quality_filters(df, text_col, "ISEAR")
    df = apply_cleaning(df, text_col, "ISEAR")
    # ISEAR is small — take ALL available rows, no cap
    df = balance_per_class(df, cap=None, source="ISEAR")
    return to_standard(df, "isear")
 
 
def load_empdial() -> pd.DataFrame:
    """
    Load EmpatheticDialogues from the Kaggle-mounted CSV.
    Expected columns: 'conv_id', 'utterance_idx', 'context' (emotion label),
                      'prompt' (user message), 'selfeval', 'tags'
    We use 'context' as the label and 'prompt' as the text.
    """
    print("\n" + "=" * 60)
    print("  Loading EmpatheticDialogues")
    print("=" * 60)
 
    df = pd.read_csv(EMPDIAL_PATH)
    print(f"  Loaded: {len(df)} rows")
    inspect_dataframe("EmpDial", df)
 
    # The 69k file uses 'context' for emotion label and 'prompt' for user text
    # Detect flexibly in case column names differ
    text_col = None
    for c in ["Situation", "prompt", "utterance", "text", "sentence"]:
        if c in df.columns:
            text_col = c
            break
    if not text_col:
        raise ValueError(f"EmpDial: cannot find text column. Columns: {df.columns.tolist()}")
 
    label_col = None
    for c in ["context", "emotion", "label"]:
        if c in df.columns:
            label_col = c
            break
    if not label_col:
        raise ValueError(f"EmpDial: cannot find label column. Columns: {df.columns.tolist()}")
 
    print(f"  Text column: '{text_col}', Label column: '{label_col}'")
 
    # EmpatheticDialogues has duplicate prompts across conversation turns.
    # Keep only utterance_idx == 1 (the opening user statement) if column exists,
    # as these most closely resemble the user rant inputs your system expects.
 
    df = apply_label_mapping(df, label_col, EMPDIAL_MAP, "EmpDial")
    df = apply_quality_filters(df, text_col, "EmpDial")
    df = apply_cleaning(df, text_col, "EmpDial")
    df = balance_per_class(df, cap=EMPDIAL_PER_CLASS, source="EmpDial")
    return to_standard(df, "empdial")


In [6]:
# COMBINE & SPLIT
def combine_and_split(go_df: pd.DataFrame,
                      isear_df: pd.DataFrame,
                      emp_df: pd.DataFrame):
    """
    1. Concatenate all three cleaned sources
    2. Deduplicate on 'text' across sources
    3. Shuffle
    4. Split 80 / 10 / 10 stratified by label
    5. Save all output CSVs
    """
    print("\n" + "=" * 60)
    print("  Combining & Splitting")
    print("=" * 60)
 
    combined = pd.concat([go_df, isear_df, emp_df], ignore_index=True)
    before = len(combined)
    combined = combined.drop_duplicates(subset=["text"])
    print(f"  Combined: {before} → {len(combined)} rows after cross-source dedup")
    print(f"  Source breakdown:\n{combined['source'].value_counts().to_string()}")
    print(f"  Label breakdown:\n{combined['label'].value_counts().to_string()}")
 
    # Stratified 80/10/10 split
    from sklearn.model_selection import train_test_split
 
    train_parts, val_parts, test_parts = [], [], []
 
    for label in combined["label"].unique():
        subset = combined[combined["label"] == label].sample(
            frac=1, random_state=RANDOM_SEED
        )
        n = len(subset)
        n_test  = max(1, int(n * TEST_RATIO))
        n_val   = max(1, int(n * VAL_RATIO))
        n_train = n - n_test - n_val
 
        train_parts.append(subset.iloc[:n_train])
        val_parts.append(subset.iloc[n_train : n_train + n_val])
        test_parts.append(subset.iloc[n_train + n_val :])
 
    train_df = pd.concat(train_parts).sample(frac=1, random_state=RANDOM_SEED)
    val_df   = pd.concat(val_parts).sample(frac=1, random_state=RANDOM_SEED)
    test_df  = pd.concat(test_parts).sample(frac=1, random_state=RANDOM_SEED)
 
    print(f"\n  Train: {len(train_df)} rows")
    print(f"  Val:   {len(val_df)} rows")
    print(f"  Test:  {len(test_df)} rows")
    print(f"\n  Train label dist:\n{train_df['label'].value_counts().to_string()}")
 
    return combined, train_df, val_df, test_df


In [7]:
# MAIN 
def main():
    print("=" * 60)
    print("  Multi-Source Emotion Dataset Pipeline")
    print("=" * 60)
 
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    # ── Load & clean each source ─────────────────────────────────────────────
    go_df    = load_goemotions()
    isear_df = load_isear()
    emp_df   = load_empdial()
 
    # ── Save individual source CSVs ──────────────────────────────────────────
    print("\n── Saving per-source CSVs ────────────────────────")
    save_csv(go_df,    "cleaned_goemotions.csv")
    save_csv(isear_df, "cleaned_isear.csv")
    save_csv(emp_df,   "cleaned_empdial.csv")
 
    # ── Combine, deduplicate, split ──────────────────────────────────────────
    combined, train_df, val_df, test_df = combine_and_split(go_df, isear_df, emp_df)
 
    # ── Save combined & split CSVs ───────────────────────────────────────────
    print("\n── Saving combined & split CSVs ──────────────────")
    save_csv(combined, "combined_all.csv")
    save_csv(train_df, "train.csv")
    save_csv(val_df,   "val.csv")
    save_csv(test_df,  "test.csv")
 
    print("\n" + "=" * 60)
    print("  All outputs saved to /kaggle/working/")
    print("  Files:")
    print("    cleaned_goemotions.csv  — GoEmotions only")
    print("    cleaned_isear.csv       — ISEAR only")
    print("    cleaned_empdial.csv     — EmpatheticDialogues only")
    print("    combined_all.csv        — All sources merged")
    print("    train.csv               — 80% split (stratified)")
    print("    val.csv                 — 10% split (stratified)")
    print("    test.csv                — 10% split (stratified)")
    print("=" * 60)
    print("\n  Columns: 'text', 'label', 'source'")
    print("  'source' column lets you trace which dataset each row came from.")
 
 
if __name__ == "__main__":
    main()

  Multi-Source Emotion Dataset Pipeline

  Loading GoEmotions
  Loaded goemotions_2.csv: 70000 rows
  Loaded goemotions_1.csv: 70000 rows
  Loaded goemotions_3.csv: 71225 rows
  Total rows after rater aggregation: 57732

── [GoEmotions] columns: ['text', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'curiosity', 'desire', 'disappointment', 'disgust', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'relief', 'remorse', 'sadness']
   shape: (57732, 23)
                                                                                                           text  admiration  amusement  anger  annoyance  approval  caring  curiosity  desire  disappointment  disgust  excitement  fear  gratitude  grief  joy  love  nervousness  optimism  pride  relief  remorse  sadness
0              "If you don't wear BROWN AND ORANGE...YOU DON'T MATTER!" We need a tshirt with that on it asap!            0          0      1          2   

ISEAR_dataset_complete.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/7516 [00:00<?, ? examples/s]

  Available splits: ['train']
  Total rows: 7516

── [ISEAR] columns: ['emotion', 'content', 'Unnamed: 2']
   shape: (7516, 3)
  emotion                                                                                                                                                                    content Unnamed: 2
0     joy  On days when I feel close to my partner and other friends.   \nWhen I feel at peace with myself and also experience a close  \ncontact with people whom I regard greatly.       None
1    fear                                                                              Every time I imagine that someone I love or I could contact a  \nserious illness, even death.       None
  Text column: 'content', Label column: 'emotion'
  Unique labels: ['joy', 'fear', 'anger', 'sadness', 'disgust', 'shame', 'guilt']
  [ISEAR] Label mapping: 7516 → 7516 rows
   Distribution:
project_label
sad        3203
angry      2145
joyful     1092
anxious    1076
  [ISEAR] Quality filter: 75